# FitMate APK Builder

一键构建 FitMate 健身助手 APK

本笔记本自动从 Windows 上传的项目文件构建 APK，约需 20-40 分钟。

In [ ]:
# ==== 第 1 步：上传项目文件 ====
import zipfile
import os
from google.colab import files

print("请选择 fitmate_project.zip 文件上传")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f"已上传: {zip_name}")

# 解压到 /content/fitmate
os.makedirs('/content/fitmate', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('/content/fitmate')
print("✅ 项目已解压到 /content/fitmate")
!ls -la /content/fitmate/

In [ ]:
# ==== 第 2 步：安装 Buildozer 及 Android 编译依赖 ====
import os
os.chdir('/content/fitmate')

print("安装系统依赖...")
!apt-get update -qq
!apt-get install -y -qq python3-pip python3-dev pkg-config libssl-dev libffi-dev \
    autoconf automake libtool cmake ccache git curl wget unzip \
    openjdk-17-jdk libncurses5 2>&1 | tail -5

print("安装 Buildozer...")
!pip install -q cython buildozer 2>&1 | tail -3

print("✅ 依赖安装完成")
!buildozer --version

In [ ]:
# ==== 第 3 步：验证 buildozer.spec ====
!cat /content/fitmate/buildozer.spec | head -30
print("\n--- buildozer.spec 已就绪 ---")

In [ ]:
# ==== 第 4 步：构建 APK ====
# 注意：首次构建需要下载 Android SDK/NDK，约 2GB
import time
start = time.time()

!buildozer android debug 2>&1

elapsed = time.time() - start
print(f"\n⏱ 构建耗时: {elapsed/60:.1f} 分钟")

In [ ]:
# ==== 第 5 步：下载 APK ====
import glob
apk_files = glob.glob('/content/fitmate/bin/*.apk')
if apk_files:
    apk_path = apk_files[0]
    print(f"APK 文件: {apk_path}")
    print(f"文件大小: {os.path.getsize(apk_path) / 1024 / 1024:.1f} MB")
    from google.colab import files
    files.download(apk_path)
else:
    print("❌ 未找到 APK 文件，请检查上方输出中的错误信息")